# ベイズ MMM（Colab 版）

上から順に、各セルの左の **▶ ボタン** を押してください。設定はセル右側のフォームで変更できます（コードは触らなくて OK）。

| ステップ | 内容 | 目安時間 |
|---|---|---|
| ① 準備 | ツールのダウンロードとインストール | 1〜2 分（初回のみ） |
| ② データ | サンプル or 自分の CSV をアップロード | すぐ |
| ③ 設定 | 列名・チャネル・計算速度など | すぐ |
| ④ 実行 | 推定してレポートを表示 | fast: 約 2〜5 分 / standard: 約 5〜15 分 |
| ⑤ シナリオ | 「TV を 1.5 倍にしたら？」を試算（再推定なし） | 数秒 |

> ⚠ Colab はブラウザを閉じる・一定時間操作しないとファイルが消えます。レポートと設定はダウンロードして保存してください。

In [ ]:
#@title ① 準備（最初に 1 回）
import os, sys, subprocess
REPO = "/content/claude-code-test"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "claude/laughing-albattani-3u9cqu",
                    "https://github.com/data-everyday/claude-code-test.git", REPO], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymc", "numpyro", "h5netcdf"], check=True)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print("✅ 準備完了")

In [ ]:
#@title ② データ
データ = "サンプルデータで試す" #@param ["サンプルデータで試す", "CSVをアップロード"]
import pandas as pd
os.makedirs("data", exist_ok=True)
if データ == "CSVをアップロード":
    from google.colab import files
    up = files.upload()
    if not up:
        raise SystemExit("ファイルが選択されませんでした")
    CSV_PATH = os.path.join("data", list(up)[0])
    with open(CSV_PATH, "wb") as f:
        f.write(list(up.values())[0])
else:
    from mmm.sample_data import generate
    CSV_PATH = "data/sample_mmm.csv"
    generate().to_csv(CSV_PATH, index=False)
df_preview = pd.read_csv(CSV_PATH)
print(f"✅ {CSV_PATH}（{len(df_preview)} 行）")
print("列名:", ", ".join(df_preview.columns))
df_preview.head()

In [ ]:
#@title ③ 設定（フォームを編集してから ▶）
#@markdown **列名**（② で表示された列名をそのまま入力）
日付の列 = "date" #@param {type:"string"}
目的変数の列 = "sales" #@param {type:"string"}
#@markdown **広告チャネル**：`名前:広告費の列名:adstock` をカンマ区切り。adstock は `geometric`（すぐ効く）/ `delayed`（遅れて効く。TV など）
チャネル = "tv:tv_spend:delayed, search:search_spend:geometric, social:social_spend:geometric" #@param {type:"string"}
#@markdown **広告以外の要因**：`列名:符号`（符号は `free` / `positive` / `negative`）。なければ空欄
コントロール = "price:negative, promo:positive" #@param {type:"string"}
#@markdown ---
#@markdown **計算速度**：`fast`＝試行用（数分）/ `standard`＝通常 / `precise`＝最終報告用（時間がかかる）
計算速度 = "fast" #@param ["fast", "standard", "precise"]
#@markdown **ホールドアウト検証**：末尾 N 週を隠して予測精度を確認（0 で省略。指定すると計算時間が約 2 倍）
ホールドアウト週数 = 0 #@param {type:"integer"}
#@markdown ---
#@markdown **モデル**
トレンド = True #@param {type:"boolean"}
季節性 = True #@param {type:"boolean"}
最大ラグ週数 = 8 #@param {type:"slider", min:2, max:13, step:1}
#@markdown ROI の事前想定（広告 1 円あたりの売上増。データが少ないチャネルはこの値に寄る）
ROI事前_中央値 = 1.0 #@param {type:"number"}
ROI事前_幅 = 1.0 #@param {type:"number"}
#@markdown ---
#@markdown **予算最適化**（総予算 0 = 実績どおり）
最適化の対象週数 = 52 #@param {type:"integer"}
総予算 = 0 #@param {type:"number"}
下限倍率 = 0.7 #@param {type:"number"}
上限倍率 = 1.3 #@param {type:"number"}

from mmm.colab import build_config, save_yaml
cfg_dict = build_config(
    csv_path=CSV_PATH, date_column=日付の列, target_column=目的変数の列,
    channels=チャネル, controls=コントロール, speed=計算速度, holdout_weeks=ホールドアウト週数,
    seasonality=季節性, trend=トレンド, max_lag=最大ラグ週数,
    roi_prior_mean=ROI事前_中央値, roi_prior_sd=ROI事前_幅,
    opt_period_weeks=最適化の対象週数, budget_lower=下限倍率, budget_upper=上限倍率,
    total_budget=総予算)
CONFIG = save_yaml(cfg_dict, "colab_config.yaml")
print("✅ 設定を保存しました:", CONFIG)
print("チャネル:", [c["name"] for c in cfg_dict["channels"]])

In [ ]:
#@title ④ 実行（推定 → レポート表示）
import subprocess, time
t0 = time.time()
proc = subprocess.Popen([sys.executable, "-m", "mmm", "run", "--config", CONFIG],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    # 進捗バーなどの細かい出力は省き、主要なログだけ表示
    if "Suppressed" in line:
        continue
    if any(k in line for k in ["サンプリング", "WARNING", "ERROR", "Error", "レポート", "ホールドアウト",
                               "split", "train", "test", "Traceback", "raise", "データ"]):
        print(line.rstrip())
proc.wait()
print(f"\n⏱ {time.time() - t0:.0f} 秒")
if proc.returncode != 0:
    raise SystemExit("❌ エラーで停止しました。上の赤字・ERROR 行を確認してください")
from IPython.display import HTML, display
display(HTML(open("mmm_output/report.html", encoding="utf-8").read()))

In [ ]:
#@title レポートと設定をダウンロード（任意）
from google.colab import files
files.download("mmm_output/report.html")
files.download(CONFIG)

In [ ]:
#@title ⑤ シナリオ試算（④ の結果を使うので再推定なし・数秒）
#@markdown `チャネル名:倍率` をカンマ区切り（例: `tv:1.5, social:0.8`）。書かないチャネルは現状維持
シナリオ名 = "TV +50%" #@param {type:"string"}
倍率 = "tv:1.5" #@param {type:"string"}
対象週数 = 52 #@param {type:"integer"}

from mmm.colab import parse_multipliers
save_yaml({"scenarios": [{"name": シナリオ名, "period_weeks": 対象週数,
                          "multipliers": parse_multipliers(倍率)}]}, "colab_scenario.yaml")
r = subprocess.run([sys.executable, "-m", "mmm", "simulate", "--config", CONFIG,
                    "--scenario-file", "colab_scenario.yaml"], capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout[-2000:], r.stderr[-2000:])
    raise SystemExit("❌ エラー")
res = pd.read_csv("mmm_output/scenarios.csv")
tot = res[res["channel"] == "TOTAL"].iloc[0]
print(f"支出 {tot['spend_change']:+,.0f} → 売上貢献 {tot['delta_contribution']:+,.0f}"
      f"（94%区間 {tot['delta_lo']:+,.0f} 〜 {tot['delta_hi']:+,.0f}、プラスになる確率 {tot['prob_delta_positive']:.0%}）")
res[["channel", "current_spend", "scenario_spend", "delta_contribution", "delta_lo", "delta_hi",
     "incremental_roi"]].round(1)